# P18b — Seed-averaged λ sweep for CORAL

Single-seed deltas are inside seed/GPU noise (~0.8pp), so we judge CORAL on
**seed-averaged** numbers. This sweep runs source-only + CORAL at several λ over
several seeds, then reports mean ± std with the keep/revert gate.

**Reuse:** your earlier P18b run is auto-reused as **seed 42** (`probs_dg` → base_s42,
`probs_align` → coral_l1_s42), so those don't retrain.

**Compute is large** (each cell ≈ one full 6-fold GNN+ConvNet arm, ~45 min on a T4).
Use the STAGED recipe below to spend the least: pick λ at one seed first, then
seed-average only the winner.

> **Final_PROJECT note (updated 2026-09-26).** All paths now come from
> `Final_PROJECT/project_paths.py` via the first code cell below — the only thing a new user may
> need to edit is the DAGHAR data path in that file (and `FINAL_PROJECT_ROOT` below, only if the
> folder isn't found automatically). **Saved cell outputs are from the original thesis run**, when
> this code lived at `/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P17_DA_LODO/Project_DA_P18b_FeatAlign`; they are kept as the record of that run.
> Note: `run_lodo.py` / the P18b runners are **not** resumable — re-running a training cell retrains and rewrites that run's output folder.
>
> ⚠️ **Historical record — do not re-run.** This notebook ran the whole P18b phase (GNN + ConvNet, source-only and CORAL arms). Final_PROJECT keeps only the ConvNet part that feeds the final model (`convnet/`, `da_baseline/convnet_predict.py`, and the source-only `base_s42/s1/s2` predictions), so the GNN/CORAL cells here would fail. To (re)produce the frozen ConvNet predictions use **`Run_Frozen_ConvNet_Colab.ipynb`** in this folder instead.

## 1. Mount & paths

In [ ]:
# ── Mount Drive & load Final_PROJECT paths (single source: Final_PROJECT/project_paths.py) ──
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob

# EDIT ONLY IF you uploaded Final_PROJECT somewhere else on your Drive
# (if this path doesn't exist, the folder is searched for automatically).
FINAL_PROJECT_ROOT = "/content/drive/MyDrive/Thesis/SO_Experiments/Final_PROJECT"

if not os.path.isfile(os.path.join(FINAL_PROJECT_ROOT, "project_paths.py")):
    hits = []
    for depth in range(0, 5):
        hits = glob.glob("/content/drive/MyDrive/" + "*/" * depth + "Final_PROJECT/project_paths.py")
        if hits: break
    assert hits, "Final_PROJECT not found on Drive — set FINAL_PROJECT_ROOT above"
    FINAL_PROJECT_ROOT = os.path.dirname(hits[0]); print("auto-detected Final_PROJECT at", FINAL_PROJECT_ROOT)
sys.path.insert(0, FINAL_PROJECT_ROOT)
import project_paths as PP            # the DAGHAR data path is set inside project_paths.py

PROJECT_PATH = PP.activate("convnet_b")   # exports DATA_PATH, checks data, cd's into this component
DATA_PATH    = PP.DATA_PATH

## 2. Setup

In [2]:
import setup_colab
setup_colab.install_deps(); assert setup_colab.verify(), "fix DATA_PATH"

Installing PyTorch Geometric ...
Done.

Verifying imports ...
  OK torch 2.11.0+cu128  CUDA=True
     GPU Tesla T4
  OK torch_geometric 2.8.0

Checking data at: /content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view
  OK KuHar              3 CSVs
  OK MotionSense        3 CSVs
  OK RealWorld-Thigh    3 CSVs
  OK RealWorld-Waist    3 CSVs
  OK UCI                3 CSVs
  OK WISDM              3 CSVs


## 3. STAGE A — pick λ at seed 42
Reuses base_s42 + coral_l1_s42 from your earlier run; only **CORAL λ=0.3 and λ=3 train
new** here (2 arms ≈ 1.5 h). Then report to see which λ looks best at seed 42.

In [3]:
!python da_baseline/run_seeds.py --data_path "$DATA_PATH" --align coral --seeds 42 --lambdas 0.3,1,3 --epochs 100 --log_every 10
!python da_baseline/run_seed_report.py

[reuse] base s42: copied /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P17_DA_LODO/Project_DA_P18b_FeatAlign/results/probs_dg -> /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P17_DA_LODO/Project_DA_P18b_FeatAlign/results/sweep/base_s42
[reuse] coral_l1 s42: copied /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P17_DA_LODO/Project_DA_P18b_FeatAlign/results/probs_align -> /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P17_DA_LODO/Project_DA_P18b_FeatAlign/results/sweep/coral_l1_s42
[1/4] base seed=42: already complete -> skip

========== [2/4] coral_l0.3 seed=42 ==========

>>> da_baseline.gnn_predict --data_path /content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view --targets all --epochs 100 --log_every 10 --seed 42 --out_dir /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P17_DA_LODO/Project_DA_P18b_FeatAlign/results/sweep/coral_l0.3_s42 --align coral --align_lambda 0.3
[GNN] P18b | mode=align=coral λ=0.3 | seed=42 | P9 recipe | device-data=/content/dri

## 4. STAGE B — seed-average baseline + the best λ
Set `BEST_LAMBDA` from Stage A (the λ with the highest seed-42 mean that doesn't tank a
fold). This trains base + that λ over seeds 1 and 2 (4 arms ≈ 3 h). Skip/extend seeds to
taste — the run is resumable.

In [4]:
BEST_LAMBDA = "1"   # <-- set from Stage A (e.g. "0.3", "1", or "3")
import os
!python da_baseline/run_seeds.py --data_path "$DATA_PATH" --align coral --seeds 1,2 --lambdas {BEST_LAMBDA} --epochs 100 --log_every 10


========== [1/4] base seed=1 ==========

>>> da_baseline.gnn_predict --data_path /content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view --targets all --epochs 100 --log_every 10 --seed 1 --out_dir /content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P17_DA_LODO/Project_DA_P18b_FeatAlign/results/sweep/base_s1
[GNN] P18b | mode=source-only baseline (DG-norm) | seed=1 | P9 recipe | device-data=/content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view | targets=['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM'] | out=/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P17_DA_LODO/Project_DA_P18b_FeatAlign/results/sweep/base_s1
[GNN] NOTE: source-only = P17 DG lower bound (target-unlabeled loaded, unused).
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
    feature_mode=hybrid, feat_dim=90

## 5. Final seed-averaged report + gate
Seed mean ± std per condition, per-fold means, Δ vs base, and the verdict.

In [5]:
!python da_baseline/run_seed_report.py


  P18b — SEED-AVERAGED sweep (fused GNN+ConvNet)

  condition        seeds       mean±std   Δ vs base  gate
  ------------------------------------------------------------
  base                 3    76.87±1.07%  (baseline)
  coral_l0.3           1    77.47±0.00%     +0.60pp  revert
  coral_l3             1    76.38±0.00%     -0.49pp  revert
  coral_l1             3    75.67±0.39%     -1.21pp  revert

  Per-fold seed-means — base vs coral_l0.3:
  fold                 base   coral_l0.3      Δpp
  ------------------------------------------------
  KuHar              65.05%       67.36%   +2.31
  MotionSense        85.03%       83.71%   -1.32
  RealWorld-Thigh    75.68%       74.29%   -1.39
  RealWorld-Waist    75.13%       75.08%   -0.05
  UCI                86.18%       85.65%   -0.53
  WISDM              74.17%       78.73%   +4.56

  Best DA condition: coral_l0.3  (77.47±0.00%, +0.60pp vs base) -> does NOT clear gate

  Gate: keep if seed-mean Δ >= +1.0pp AND no fold seed-mean drops >

---
### Reading it
- A condition **KEEPS** only if its seed-mean beats base by **≥ +1.0pp** with **no fold < −2pp**.
- If CORAL clears the gate at some λ → carry it into **P18c** (DANN source→target).
- If nothing clears it on seed-averaged numbers → CORAL is genuinely neutral here; go to
  **P18c**, or reconsider scope (the WISDM 4-of-6-class shift is the recurring drag —
  class-conditional alignment in P19 would target that).

### Full grid (optional, expensive)
To run everything at once instead of staged:
```
!python da_baseline/run_seeds.py --data_path "$DATA_PATH" --align coral --seeds 42,1,2 --lambdas 0.3,1,3 --epochs 100 --log_every 10
!python da_baseline/run_seed_report.py
```